# Retrieval Experimentation and Analysis

This notebook reports retrieval accuracy (`Recall@K`, `HitRate@K`) and latency (mean/p50/p95 in milliseconds).

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

def find_project_root(start: Path) -> Path:
    markers = ["requirements.txt", "dataset", "experiments"]
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if all((candidate / marker).exists() for marker in markers):
            return candidate
    raise RuntimeError(
        "Could not locate project root from current working directory. "
        "Open notebook from project root or set cwd inside repository."
    )

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from experiments.retrieval_metrics import evaluate_retrieval, load_jds, load_resumes

In [ ]:
RESUME_DIR = PROJECT_ROOT / "dataset" / "resumes"
JD_DIR = PROJECT_ROOT / "dataset" / "jds"
K_VALUES = [1, 3, 5, 10]
LATENCY_REPEATS = 25

resumes = load_resumes(str(RESUME_DIR))
jobs = load_jds(str(JD_DIR))

if not resumes:
    raise ValueError(f"No resumes found in {RESUME_DIR}")
if not jobs:
    raise ValueError(f"No JDs found in {JD_DIR}")

print(f"Project root: {PROJECT_ROOT}")
print(f"Loaded resumes: {len(resumes)}")
print(f"Loaded JDs: {len(jobs)}")

## 1) Accuracy + Latency (strict relevance)

`all_required` relevance mode considers a resume relevant if all required JD skills are present and minimum experience is satisfied.

In [ ]:
strict_results = evaluate_retrieval(
    resumes=resumes,
    jobs=jobs,
    k_values=K_VALUES,
    relevance_mode="all_required",
    latency_repeats=LATENCY_REPEATS,
)

metrics_df = pd.DataFrame(strict_results["metrics"])
latency_df = pd.DataFrame(strict_results["latency_rows"])
summary = strict_results["summary"]

required_metric_cols = {"k", "recall_at_k", "hit_rate_at_k"}
missing_metric_cols = required_metric_cols - set(metrics_df.columns)
if missing_metric_cols:
    raise ValueError(f"Missing metric columns: {sorted(missing_metric_cols)}")

if "latency_ms" not in latency_df.columns:
    raise ValueError("Missing latency column: latency_ms")

display(metrics_df)
display(latency_df.head())
summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(metrics_df["k"], metrics_df["recall_at_k"], marker="o", label="Recall@K")
axes[0].plot(metrics_df["k"], metrics_df["hit_rate_at_k"], marker="o", label="HitRate@K")
axes[0].set_title("Retrieval Accuracy")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Score")
axes[0].set_ylim(0, 1.05)
axes[0].legend()

axes[1].hist(latency_df["latency_ms"], bins=10, color="#4F81BD", edgecolor="white")
axes[1].set_title("Latency Distribution (per JD)")
axes[1].set_xlabel("Latency (ms)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## 2) Relevance Sensitivity Experiment

Compare three relevance definitions: `all_required`, `half_required`, and `any_required`.

In [ ]:
modes = ["all_required", "half_required", "any_required"]
rows = []

for mode in modes:
    result = evaluate_retrieval(
        resumes=resumes,
        jobs=jobs,
        k_values=[5],
        relevance_mode=mode,
        latency_repeats=LATENCY_REPEATS,
    )
    metric = result["metrics"][0]
    rows.append(
        {
            "relevance_mode": mode,
            "recall_at_5": metric["recall_at_k"],
            "hit_rate_at_5": metric["hit_rate_at_k"],
            "avg_latency_ms": result["summary"]["avg_latency_ms"],
            "p95_latency_ms": result["summary"]["p95_latency_ms"],
        }
    )

sensitivity_df = pd.DataFrame(rows)
display(sensitivity_df)

## 3) Final Reporting Block

Use this output in report or demo notes.

In [ ]:
report = {
    "dataset": {
        "resumes": len(resumes),
        "jds": len(jobs),
    },
    "accuracy": metrics_df.to_dict(orient="records"),
    "latency_summary_ms": summary,
}

report